### Unmix_Gui_Multi ex 
──────────────────────────────────────────────────────────
• Bands are chosen by wavelength range (nm_min..nm_max; from XML)
• Endmembers: min 2 (required), up to 4 total via “Add CSV…”
• Output labels (fluor names) strictly preserved (folder & filename)
• Unmixing algorithm selectable: NNLS or FCLSU (sum-to-one & nonnegativity)

Deps: numpy, pandas, tifffile, scipy (optimize), tkinter

In [13]:
import os
import re
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import tkinter as tk
from tkinter import ttk, filedialog, messagebox

import numpy as np
import pandas as pd
import tifffile as tiff
from scipy.optimize import nnls, minimize
from xml.etree import ElementTree as ET

# ---------- Helpers ----------

LOOP_FILE_REGEX = re.compile(
    r"^Extracted_?loop(?P<loop>\d+)_(?P<wl>\d{3})nm_object_(?P<exp>\d+ms)\.(?:tif|tiff)$",
    re.IGNORECASE
)

def load_tiff_stack(path: Path) -> np.ndarray:
    arr = tiff.imread(str(path))
    return arr if arr.ndim == 3 else arr[None, ...]  # (bands,H,W)

def parse_xml_wavelengths(xml_path: Path) -> List[float]:
    root = ET.parse(xml_path).getroot()
    ws = []
    for band in root.findall(".//band"):
        try:
            wl = float(band.attrib.get("wavelength_nm", "0"))
        except ValueError:
            continue
        if wl > 0:
            ws.append(wl)
    return ws

def indices_for_nm_window(xml_wavelengths: List[float], nm_min: float, nm_max: float) -> List[int]:
    return [i for i, w in enumerate(xml_wavelengths) if nm_min <= w <= nm_max]

def read_endmember_csv(csv_path: Path, row_index: int = 1) -> Tuple[np.ndarray, np.ndarray]:
    """Columns must be wavelengths (float-able); returns (wavelengths, spectrum row)."""
    df = pd.read_csv(csv_path)
    wl = np.array([float(c) for c in df.columns], dtype=float)
    y  = df.iloc[row_index].to_numpy(dtype=float)
    if y.shape[0] != wl.shape[0]:
        raise ValueError(f"Length mismatch in {csv_path.name} (cols={wl.shape[0]}, row len={y.shape[0]})")
    return wl, y

def align_endmember_to_xml_subset(em_wl: np.ndarray, em_y: np.ndarray, xml_wl_subset: List[float], tol: float = 0.11) -> np.ndarray:
    """
    For each XML wavelength in subset, pick nearest EM wavelength within tol (nm),
    otherwise linear-interpolate over endmember wavelengths.
    """
    em_wl = np.asarray(em_wl, dtype=float)
    em_y  = np.asarray(em_y, dtype=float)
    target = np.asarray(xml_wl_subset, dtype=float)
    out = np.empty_like(target, dtype=float)
    for i, w in enumerate(target):
        j = int(np.argmin(np.abs(em_wl - w)))
        if abs(em_wl[j] - w) <= tol:
            out[i] = em_y[j]
        else:
            out[i] = np.interp(w, em_wl, em_y)
    return out

def solve_nnls_stack(stack: np.ndarray, band_idx: List[int], E: np.ndarray) -> np.ndarray:
    """
    NNLS per pixel.
    stack: (bands,H,W), band_idx: list[int], E: (len(band_idx), M)
    return maps: (M,H,W)
    """
    sel = stack[band_idx, :, :]
    nb, H, W = sel.shape
    pixels = sel.reshape(nb, -1).T
    M = E.shape[1]
    abund = np.empty((pixels.shape[0], M), dtype=np.float32)
    for k, px in enumerate(pixels):
        abund[k, :] = nnls(E, px)[0]
    return abund.T.reshape(M, H, W).astype(np.float32)

def solve_fclsu_stack(stack: np.ndarray, band_idx: List[int], E: np.ndarray) -> np.ndarray:
    """
    FCLSU via SLSQP per pixel: min 0.5||Ea - x||^2 s.t. a>=0, sum(a)=1.
    If solver fails, falls back to NNLS for that pixel then renormalizes sum to 1 (if >0).
    """
    sel = stack[band_idx, :, :]
    nb, H, W = sel.shape
    pixels = sel.reshape(nb, -1).T
    M = E.shape[1]
    abund = np.empty((pixels.shape[0], M), dtype=np.float32)

    # Precompute for speed
    ET_E = E.T @ E
    ET   = E.T

    bounds = [(0.0, None)] * M
    cons = dict(type='eq', fun=lambda a: np.sum(a) - 1.0)

    # warm start: uniform on simplex
    a0 = np.full(M, 1.0 / M, dtype=float)

    def obj(a, x):
        r = E @ a - x
        return 0.5 * float(r @ r)

    def jac(a, x):
        # gradient = E^T (E a - x)
        return (ET_E @ a - ET @ x)

    for k, x in enumerate(pixels):
        res = minimize(obj, a0, args=(x,), jac=jac, method='SLSQP',
                       bounds=bounds, constraints=[cons],
                       options=dict(maxiter=200, ftol=1e-9, disp=False))
        if not res.success:
            # fallback: NNLS then simplex renorm (if positive)
            a_nnls = nnls(E, x)[0]
            s = a_nnls.sum()
            if s > 0:
                a = a_nnls / s
            else:
                a = a0.copy()
        else:
            a = res.x
        abund[k, :] = a.astype(np.float32)

    return abund.T.reshape(M, H, W).astype(np.float32)

# ---------- GUI blocks ----------

DEFAULT_WL = (488, 561, 647, 730)

class EndmembersUpTo4(ttk.LabelFrame):
    """Endmembers: 2 required, up to 4 total. (path,label) pairs; labels drive outputs."""
    def __init__(self, master):
        super().__init__(master, text="Endmembers (min 2, max 4)")
        self.items: List[Tuple[Path, str]] = []
        self._build()

    def _build(self):
        p = {"padx":6, "pady":3}
        self.lb = tk.Listbox(self, height=5, width=62)
        self.lb.grid(row=0, column=0, columnspan=5, sticky="we", **p)

        ttk.Button(self, text="Add CSV…", command=self._add_csv).grid(row=1, column=0, **p)
        ttk.Button(self, text="Edit label", command=self._edit_label).grid(row=1, column=1, **p)
        ttk.Button(self, text="Remove", command=self._remove).grid(row=1, column=2, **p)
        ttk.Button(self, text="Clear", command=self._clear).grid(row=1, column=3, **p)
        ttk.Label(self, text="Tip: Labels should be fluor names (e.g., Cy3, A594, Alexa488, Cy5.5).").grid(
            row=2, column=0, columnspan=5, sticky="w", padx=6
        )

    def _add_csv(self):
        if len(self.items) >= 4:
            messagebox.showwarning("Endmembers", "Maximum of 4 endmembers is supported.")
            return
        files = filedialog.askopenfilenames(title="Add endmember CSVs",
                                            filetypes=[("CSV","*.csv"),("All files","*.*")])
        for f in files:
            if len(self.items) >= 4:
                break
            p = Path(f)
            label = p.stem
            self.items.append((p, label))
            self.lb.insert(tk.END, f"{label}  —  {p}")

    def _edit_label(self):
        sel = self.lb.curselection()
        if not sel:
            return
        idx = sel[0]
        p, old_label = self.items[idx][0], self.items[idx][1]
        top = tk.Toplevel(self); top.title("Edit label")
        var = tk.StringVar(value=old_label)
        ttk.Entry(top, textvariable=var, width=36).grid(row=0, column=0, padx=8, pady=8)
        def ok():
            new_label = var.get().strip() or p.stem
            self.items[idx] = (p, new_label)
            self.lb.delete(idx)
            self.lb.insert(idx, f"{new_label}  —  {p}")
            top.destroy()
        ttk.Button(top, text="OK", command=ok).grid(row=1, column=0, padx=8, pady=8)

    def _remove(self):
        sel = list(self.lb.curselection())
        sel.reverse()
        for i in sel:
            self.lb.delete(i)
            self.items.pop(i)

    def _clear(self):
        self.lb.delete(0, tk.END)
        self.items.clear()

    def get_items(self) -> List[Tuple[Path,str]]:
        return list(self.items)

class ChannelUI(ttk.LabelFrame):
    def __init__(self, master, wavelength: int):
        super().__init__(master, text=f"ex{wavelength}")
        self.wl = wavelength
        self.var_folder = tk.StringVar(value="")
        self.var_loops  = tk.IntVar(value=300)
        self.var_nm_min = tk.DoubleVar(value=575.0 if wavelength==561 else (675.0 if wavelength==647 else (500.0 if wavelength==488 else 750.0)))
        self.var_nm_max = tk.DoubleVar(value=840.0 if wavelength==561 else (900.0 if wavelength in (647,730) else 700.0))
        self.endm = EndmembersUpTo4(self)
        self._build()

    def _build(self):
        p = {"padx":6, "pady":3}
        ttk.Label(self, text="Extracted TIFF folder").grid(row=0, column=0, sticky="e", **p)
        ttk.Entry(self, textvariable=self.var_folder, width=54).grid(row=0, column=1, sticky="we", **p)
        ttk.Button(self, text="Browse", command=self._choose_folder).grid(row=0, column=2, **p)

        ttk.Label(self, text="Loops (N)").grid(row=1, column=0, sticky="e", **p)
        ttk.Entry(self, textvariable=self.var_loops, width=8).grid(row=1, column=1, sticky="w", **p)

        ttk.Label(self, text="Unmix nm range").grid(row=2, column=0, sticky="e", **p)
        ttk.Entry(self, textvariable=self.var_nm_min, width=10).grid(row=2, column=1, sticky="w", **p)
        ttk.Label(self, text="to").grid(row=2, column=1, padx=(100,0), sticky="w")
        self.e_nm_max = ttk.Entry(self, textvariable=self.var_nm_max, width=10)
        self.e_nm_max.place(x=260, y=72)

        self.endm.grid(row=3, column=0, columnspan=3, sticky="we", **p)

    def _choose_folder(self):
        d = filedialog.askdirectory(title=f"Choose Extracted folder for ex{self.wl}")
        if d:
            self.var_folder.set(d)

    def params(self) -> Dict:
        return dict(
            wl=self.wl,
            folder=Path(self.var_folder.get().strip()) if self.var_folder.get().strip() else None,
            loops=int(self.var_loops.get()),
            nm_min=float(self.var_nm_min.get()),
            nm_max=float(self.var_nm_max.get()),
            endmembers=self.endm.get_items(),
        )

class App(ttk.Frame):
    def __init__(self, master):
        super().__init__(master)
        master.title("NNLS / FCLSU Unmixing — nm-range, up to 4 endmembers (488/561/647/730)")
        master.protocol("WM_DELETE_WINDOW", master.destroy)
        self.grid(padx=10, pady=10, sticky="nsew")

        # Global settings
        self.var_out = tk.StringVar(value=os.path.join(os.path.expanduser("~"), "unmixed_output"))
        self.var_xml = tk.StringVar(value="")
        self.var_algo = tk.StringVar(value="NNLS")  # <—— algorithm chooser
        self._build_header()

        # Channels
        self.channels: Dict[int, ChannelUI] = {}
        row = 1
        for wl in DEFAULT_WL:
            ch = ChannelUI(self, wl)
            ch.grid(row=row, column=0, sticky="nsew", pady=(6,2))
            self.channels[wl] = ch
            row += 1

        ctrl = ttk.Frame(self)
        ctrl.grid(row=row, column=0, sticky="e", pady=(8,0))
        ttk.Button(ctrl, text="Run", command=self.on_run).grid(row=0, column=0, padx=6)
        ttk.Button(ctrl, text="Quit", command=self.master.destroy).grid(row=0, column=1, padx=6)

        self.status = tk.StringVar(value="Ready.")
        ttk.Label(self, textvariable=self.status).grid(row=row+1, column=0, sticky="w", pady=(6,0))

    def _build_header(self):
        f = ttk.LabelFrame(self, text="Global Settings")
        f.grid(row=0, column=0, sticky="ew")
        ttk.Label(f, text="Output root").grid(row=0, column=0, padx=6, pady=6, sticky="e")
        ttk.Entry(f, textvariable=self.var_out, width=48).grid(row=0, column=1, padx=6, pady=6)
        ttk.Button(f, text="Browse", command=self._choose_out).grid(row=0, column=2, padx=6, pady=6)

        ttk.Label(f, text="Single XML (optional, used for all)").grid(row=1, column=0, padx=6, pady=6, sticky="e")
        ttk.Entry(f, textvariable=self.var_xml, width=48).grid(row=1, column=1, padx=6, pady=6)
        ttk.Button(f, text="Browse", command=self._choose_xml).grid(row=1, column=2, padx=6, pady=6)

        # Algorithm chooser
        ttk.Label(f, text="Unmixing algorithm").grid(row=2, column=0, padx=6, pady=6, sticky="e")
        algo_box = ttk.Combobox(f, textvariable=self.var_algo, values=["NNLS", "FCLSU (sum-to-one)"], width=30, state="readonly")
        algo_box.grid(row=2, column=1, padx=6, pady=6, sticky="w")
        algo_box.current(0)

    def _choose_out(self):
        d = filedialog.askdirectory(title="Choose output root")
        if d:
            self.var_out.set(d)

    def _choose_xml(self):
        f = filedialog.askopenfilename(title="Choose XML", filetypes=[("XML","*.xml"),("All files","*.*")])
        if f:
            self.var_xml.set(f)

    def on_run(self):
        out_root = Path(self.var_out.get().strip())
        if not out_root:
            messagebox.showerror("Output", "Please choose an output root.")
            return
        global_xml = Path(self.var_xml.get().strip()) if self.var_xml.get().strip() else None
        algo = self.var_algo.get()

        any_ok = any(ch.params()["folder"] and len(ch.params()["endmembers"]) >= 2 for ch in self.channels.values())
        if not any_ok:
            messagebox.showwarning("No input", "For at least one channel, set folder and add ≥2 endmember CSVs.")
            return

        try:
            for wl, ch in self.channels.items():
                p = ch.params()
                folder: Optional[Path] = p["folder"]
                if not folder:
                    continue
                em_list = p["endmembers"]
                if len(em_list) < 2:
                    messagebox.showwarning(f"ex{wl}", "At least two endmembers are required.")
                    continue
                if len(em_list) > 4:
                    messagebox.showwarning(f"ex{wl}", "Maximum of four endmembers is supported; extra will be ignored.")
                    em_list = em_list[:4]

                nm_min, nm_max = float(p["nm_min"]), float(p["nm_max"])
                loops = max(1, int(p["loops"]))

                # XML: global or sidecar
                xml_path: Optional[Path] = None
                if global_xml and global_xml.exists():
                    xml_path = global_xml
                else:
                    candidate = None
                    for f in folder.iterdir():
                        if f.is_file() and LOOP_FILE_REGEX.match(f.name):
                            sidecar = Path(str(f) + ".xml")
                            if sidecar.exists():
                                candidate = sidecar
                                break
                    if candidate is None:
                        messagebox.showerror(f"ex{wl}", "XML not found (global not set and no sidecar .xml).")
                        continue
                    xml_path = candidate

                xml_wls = parse_xml_wavelengths(xml_path)
                if not xml_wls:
                    messagebox.showwarning(f"ex{wl}", "No wavelengths in XML.")
                    continue
                band_idx_full = indices_for_nm_window(xml_wls, nm_min, nm_max)
                if not band_idx_full:
                    messagebox.showwarning(f"ex{wl}", f"No bands within {nm_min}–{nm_max} nm.")
                    continue

                # Build endmember matrix aligned to nm window
                E_cols, labels = [], []
                xml_subset = [xml_wls[i] for i in band_idx_full]
                for (csv_path, lbl) in em_list:
                    em_wl, em_y = read_endmember_csv(csv_path, row_index=1)
                    em_y_sel = align_endmember_to_xml_subset(em_wl, em_y, xml_subset, tol=0.11)
                    E_cols.append(em_y_sel)
                    labels.append(lbl)
                E = np.stack(E_cols, axis=1)  # (nbands, M)
                M = E.shape[1]

                base_out = out_root / f"ex{wl}"
                for lbl in labels:
                    (base_out / lbl).mkdir(parents=True, exist_ok=True)

                print(f"\n===== ex{wl} | nm={nm_min}–{nm_max} | bands={len(band_idx_full)} | algo={algo} | endmembers={labels} =====")

                # Choose solver
                solver = solve_nnls_stack if algo.startswith("NNLS") else solve_fclsu_stack

                # Process loops
                for i in range(1, loops + 1):
                    tif_path = None
                    for f in folder.iterdir():
                        if not f.is_file():
                            continue
                        m = LOOP_FILE_REGEX.match(f.name)
                        if m and int(m.group("wl")) == wl and int(m.group("loop")) == i:
                            tif_path = f; break
                    if tif_path is None:
                        print(f"⚠ Missing loop{i} for ex{wl}")
                        continue

                    try:
                        stack = load_tiff_stack(tif_path)
                        idx = [j for j in band_idx_full if 0 <= j < stack.shape[0]]
                        if not idx:
                            print(f"⚠ No usable bands in stack for {tif_path.name}")
                            continue

                        # Align E if we clipped indices
                        if len(idx) != len(band_idx_full):
                            keep = [band_idx_full.index(j) for j in idx]
                            E_use = E[keep, :]
                        else:
                            E_use = E

                        maps = solver(stack, idx, E_use)  # (M,H,W)

                        # Extract loop number, wavelength, and exposure time from original filename
                        m = LOOP_FILE_REGEX.match(tif_path.name)
                        loop_num = m.group("loop")
                        wl_str = m.group("wl")
                        exp_str = m.group("exp")

                        for m_idx, lbl in enumerate(labels):
                            # New filename format: {label}_Unmixed_loop{N}_{WL}nm_object_{EXP}.tif
                            out_name = f"{lbl}_Unmixed_loop{loop_num}_{wl_str}nm_object_{exp_str}.tif"
                            out_path = (base_out / lbl) / out_name
                            tiff.imwrite(str(out_path), maps[m_idx], photometric="minisblack")
                        print(f"  ✓ {tif_path.name}")

                    except Exception as e:
                        print(f"  ❌ {tif_path.name} -> {e}")

            self.status.set("Done. Check the output folders.")
            messagebox.showinfo("Finished", "Unmixing completed.")

        except Exception as e:
            self.status.set("Error occurred.")
            messagebox.showerror("Error", str(e))

def main():
    root = tk.Tk()
    try:
        root.call("source", "azure.tcl"); root.call("set_theme", "light")
    except Exception:
        pass
    App(root); root.mainloop()

if __name__ == "__main__":
    main()
